# Notebook del modelo

Resumen operativo del pipeline secuencial actual en su versión v2:

- parte de un dataset sintético crudo con etiquetas temporales
- construye ventanas deslizantes por `sample_id`
- entrena `LSTM` y `GRU`
- compara ambos modelos
- conserva el ganador y sus artefactos


## Arquitectura actual

La partición se estratifica por `target_global` para no romper el balance de clases a nivel de serie,
pero la predicción se realiza sobre ventanas etiquetadas con `target`, que refleja la fase temporal
de cada tramo.


In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "config" / "config.yaml").exists():
    ROOT = ROOT.parent

summary = json.loads((ROOT / "models" / "metrics" / "metrics_summary.json").read_text(encoding="utf-8"))
comparison = pd.read_csv(ROOT / "models" / "metrics" / "model_comparison.csv")
display(comparison)
print("Modelo ganador:", summary["best_model_name"])
print("Metrica de seleccion:", summary["selection_metric"])
print("Valor ganador:", summary["best_model_metric"])


,model_name,selection_metric,selection_value,accuracy,balanced_accuracy,f1_macro,precision_macro,recall_macro,log_loss,predictions_file
0,gru,f1_macro,0.943587,0.941441,0.945202,0.943587,0.942235,0.945202,0.183020,data\predictions\holdout_predictions_gru.csv
1,lstm,f1_macro,0.932763,0.930180,0.934612,0.932763,0.931277,0.934612,0.209667,data\predictions\holdout_predictions_lstm.csv


Modelo ganador: gru
Metrica de seleccion: f1_macro
Valor ganador: 0.9435867107634568


## Búsqueda de hiperparámetros

La exploración queda registrada para ambas arquitecturas y permite revisar qué configuraciones fueron
evaluadas y cómo evolucionó el entrenamiento por época.


In [3]:
lstm_search = pd.read_csv(ROOT / "models" / "metrics" / "lstm_search_summary.csv")
gru_search = pd.read_csv(ROOT / "models" / "metrics" / "gru_search_summary.csv")
lstm_hist = pd.read_csv(ROOT / "models" / "metrics" / "lstm_training_history.csv")
gru_hist = pd.read_csv(ROOT / "models" / "metrics" / "gru_training_history.csv")
score_col = "selection_value" if "selection_value" in lstm_search.columns else "f1_macro"

print("Top configuraciones LSTM")
display(lstm_search.sort_values(score_col, ascending=False).head())
print("Top configuraciones GRU")
display(gru_search.sort_values(score_col, ascending=False).head())
print("Historial LSTM")
display(lstm_hist.tail())
print("Historial GRU")
display(gru_hist.tail())


Top configuraciones LSTM


,model_type,trial_id,selection_metric,selection_value,val_accuracy,val_balanced_accuracy,val_f1_macro,val_precision_macro,val_recall_macro,val_log_loss,params
0,lstm,2,f1_macro,0.946649,0.944745,0.946246,0.946649,0.947170,0.946246,0.176173,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."
1,lstm,4,f1_macro,0.946123,0.944745,0.946071,0.946123,0.946397,0.946071,0.174312,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."
2,lstm,6,f1_macro,0.944067,0.942342,0.945165,0.944067,0.944888,0.945165,0.180724,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."
3,lstm,7,f1_macro,0.943760,0.941742,0.942751,0.943760,0.944898,0.942751,0.167537,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."
4,lstm,5,f1_macro,0.942338,0.940541,0.942398,0.942338,0.942570,0.942398,0.168643,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."


Top configuraciones GRU


,model_type,trial_id,selection_metric,selection_value,val_accuracy,val_balanced_accuracy,val_f1_macro,val_precision_macro,val_recall_macro,val_log_loss,params
0,gru,3,f1_macro,0.954294,0.953153,0.954509,0.954294,0.954315,0.954509,0.145483,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."
1,gru,4,f1_macro,0.953092,0.951351,0.953925,0.953092,0.952365,0.953925,0.142991,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."
2,gru,5,f1_macro,0.951110,0.948949,0.951174,0.951110,0.951085,0.951174,0.162831,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."
3,gru,6,f1_macro,0.949757,0.948348,0.949371,0.949757,0.950263,0.949371,0.147279,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."
4,gru,7,f1_macro,0.949440,0.947748,0.949356,0.949440,0.949528,0.949356,0.148947,"{""batch_size"": 64, ""bidirectional"": false, ""dr..."


Historial LSTM


,model_type,trial_id,epoch,train_loss,val_loss,val_accuracy,val_balanced_accuracy,val_f1_macro,val_precision_macro,val_recall_macro,val_log_loss
93,lstm,8,5,0.104647,0.171915,0.938138,0.939641,0.939772,0.939938,0.939641,0.178669
94,lstm,8,6,0.086710,0.185979,0.933934,0.935745,0.936411,0.938176,0.935745,0.191539
95,lstm,8,7,0.079440,0.173803,0.934535,0.938153,0.936772,0.935667,0.938153,0.183222
96,lstm,8,8,0.075439,0.169510,0.938138,0.940736,0.939577,0.938668,0.940736,0.177130
97,lstm,8,9,0.070126,0.220498,0.932733,0.934121,0.935094,0.936255,0.934121,0.225500


Historial GRU


,model_type,trial_id,epoch,train_loss,val_loss,val_accuracy,val_balanced_accuracy,val_f1_macro,val_precision_macro,val_recall_macro,val_log_loss
95,gru,8,11,0.037836,0.189984,0.939339,0.941113,0.940421,0.939931,0.941113,0.198173
96,gru,8,12,0.033993,0.193905,0.943544,0.945495,0.945696,0.945903,0.945495,0.199968
97,gru,8,13,0.027322,0.217227,0.929129,0.929866,0.931854,0.936531,0.929866,0.219361
98,gru,8,14,0.022938,0.218278,0.937538,0.939280,0.938931,0.938830,0.939280,0.231805
99,gru,8,15,0.017764,0.222496,0.939339,0.941495,0.941159,0.942049,0.941495,0.232164


## Evidencia temporal

El dataset actual incorpora `target_global` como referencia de la serie completa y `target` como
etiqueta temporal efectiva. Esto permite entrenar sobre ventanas con transición de estado más realista.


In [4]:
raw_df = pd.read_csv(ROOT / "data" / "raw" / "dataset_infestacion_cereales_sintetico.csv")
cols = [c for c in ["sample_id", "target_global", "target", "phase_name", "healthy_until_step", "transition_to_insect_end_step", "transition_to_moho_end_step"] if c in raw_df.columns]
display(raw_df[cols].head(10))

print("Distribucion target_global:")
print(raw_df["target_global"].value_counts().sort_index())
print()
print("Distribucion target temporal:")
print(raw_df["target"].value_counts().sort_index())


,sample_id,target_global,target,phase_name,healthy_until_step,transition_to_insect_end_step,transition_to_moho_end_step
0,S_0_0000,0,0,sano,68,NaN,NaN
1,S_0_0000,0,0,sano,68,NaN,NaN
2,S_0_0000,0,0,sano,68,NaN,NaN
3,S_0_0000,0,0,sano,68,NaN,NaN
4,S_0_0000,0,0,sano,68,NaN,NaN
5,S_0_0000,0,0,sano,68,NaN,NaN
6,S_0_0000,0,0,sano,68,NaN,NaN
7,S_0_0000,0,0,sano,68,NaN,NaN
8,S_0_0000,0,0,sano,68,NaN,NaN
9,S_0_0000,0,0,sano,68,NaN,NaN


Distribucion target_global:
target_global
0    48000
1    48000
2    48000
Name: count, dtype: int64

Distribucion target temporal:
target
0    59947
1    47983
2    36070
Name: count, dtype: int64


## Criterios de aceptación

Además de elegir el ganador por `f1_macro`, el flujo de evaluación escribe un chequeo formal de umbrales
mínimos para asegurar que el resultado final cumple las condiciones operativas definidas en configuración.


In [5]:
acceptance = pd.DataFrame(summary.get("acceptance_check", {}).get("checks", []))
if not acceptance.empty:
    display(acceptance)
    print("Resultado global:", summary["acceptance_check"].get("overall_passed"))
else:
    print("No hay chequeos de aceptacion disponibles.")


,name,metric,operator,required,actual,passed
0,f1_macro_min,f1_macro,>=,0.90,0.943587,True
1,recall_macro_min,recall_macro,>=,0.90,0.945202,True
2,accuracy_min,accuracy,>=,0.90,0.941441,True
3,log_loss_max,log_loss,<=,0.20,0.183020,True
4,class_recall_min.moho_critico,moho_critico.recall,>=,0.95,0.988506,True


Resultado global: True
